# NTv3 functional-tracks model (JAX)

The NTv3 tracks model predicts 7,362 functional-genomics tracks (bigwig signal)
at single-base resolution, conditioned on the species. It is loaded from the
Flax weights in `InstaDeepAI/NTv3_tracks`. See
[docs/nucleotide_transformer_v3.md](../../docs/nucleotide_transformer_v3.md).

In [ ]:
import gzip
import os
import re

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import requests
import seaborn as sns
from pyfaidx import Fasta

from nucleotide_transformer_v3.pretrained import load_tracks_model

## 1. Load the model

Besides the sequence tokenizer, the loader returns one tokenizer per condition.
This model has a single one, `species`, whose ids are the only valid way to say
which species a sequence comes from.

In [ ]:
model, tokenizer, (species_tokenizer,) = load_tracks_model("NTv3_tracks")

config = model.config
print(f"Sequence vocabulary: {tokenizer.vocabulary_size} tokens (soft-mask aware)")
print(f"Species: {species_tokenizer.standard_tokens}")
print(f"Bigwig tracks: {config.number_bigwig_tracks}")
print(f"Output crop: central {config.keep_target_center_fraction:.1%} of the input")
print("Tracks per assembly:")
for assembly_name, track_ids in config.bigwigs_per_file_assembly.items():
    print(f"  {assembly_name}: {len(track_ids)}")

## 2. Get a sequence

**Keep the soft-masking.** The 16-token vocabulary gives lowercase bases their
own ids, and the model was trained on repeat-masked assemblies. It has no IUPAC
ambiguity codes, though, and the fast `batch_np_tokenize` stops at the first
one (a silently truncated array with NumPy < 2.3, a `ValueError` with 2.3+), so
map those to `N` or `n`, keeping their case.

In [ ]:
def download_ucsc_chrom_fasta(chrom: str, assembly: str, out_dir: str | None = None) -> str:
    """Download a single chromosome FASTA from UCSC and return its local path."""
    out_dir = out_dir or f"./{assembly}"
    os.makedirs(out_dir, exist_ok=True)
    gz_path = os.path.join(out_dir, f"{chrom}.fa.gz")
    fa_path = os.path.join(out_dir, f"{chrom}.fa")
    if os.path.exists(fa_path):
        print(f"Using cached FASTA: {fa_path}")
        return fa_path

    url = f"https://hgdownload.soe.ucsc.edu/goldenPath/{assembly}/chromosomes/{chrom}.fa.gz"
    print(f"Downloading: {url}")
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(gz_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
    with gzip.open(gz_path, "rb") as fin, open(fa_path, "wb") as fout:
        fout.write(fin.read())
    return fa_path


def fetch_window_sequence(chrom: str, start: int, end: int, fasta_path: str) -> str:
    """Fetch [start, end) from the FASTA, case preserved."""
    return Fasta(fasta_path, rebuild=False)[chrom][start:end].seq

In [ ]:
# 131,072 bp keeps this runnable on a laptop CPU. The model was released at
# 131,072 bp; NTv3_tracks was trained on inputs of 16 kb to 1 Mb.
assembly = "hg38"
chrom = "chr19"
start = 6_700_000
end = 6_831_072

window_len = end - start
assert window_len % 128 == 0, f"Window length ({window_len}) must be a multiple of 128"

fasta_path = download_ucsc_chrom_fasta(chrom, assembly)
raw_seq = fetch_window_sequence(chrom, start, end, fasta_path)
softmasked = sum(c.islower() for c in raw_seq) / len(raw_seq)
print(f"Window: {chrom}:{start:,}-{end:,} ({window_len:,} bp), {softmasked:.1%} soft-masked")

seq = re.sub(r"[^ACGTNacgtn]", lambda m: "n" if m.group().islower() else "N", raw_seq)
assert sum(c.islower() for c in seq) == sum(c.islower() for c in raw_seq)
tokens = jnp.asarray(tokenizer.batch_np_tokenize([seq]))
assert tokens.shape[1] == window_len, "tokenization truncated the sequence"
softmasked_ids = int((tokens >= 11).sum())
print(f"Tokens: {tokens.shape}, {softmasked_ids:,} soft-masked token ids (11-15)")

## 3. Choose tracks and species

A track's index is its position in `bigwigs_per_file_assembly[assembly]`, and
that is also its column in the output. Only the first
`len(bigwigs_per_file_assembly[assembly])` columns belong to an assembly; the
rest are other assemblies' columns.

**Always pass `track_indices`.** Without it the head computes all 7,362 tracks
at every base: at the released 1 Mb window that output alone is ~10.8 GiB per
sequence in float32.

In [ ]:
tracks_to_plot = {
    "K562 RNA-seq": "ENCSR056HPM",
    "K562 DNase": "ENCSR921NMD",
    "K562 H3K4me3": "ENCSR000DWD",
    "K562 CTCF": "ENCSR000AKO",
    "HepG2 RNA-seq": "ENCSR561FEE_P",
    "HepG2 DNase": "ENCSR000EJV",
    "HepG2 H3K4me3": "ENCSR000AMP",
    "HepG2 CTCF": "ENCSR000BIE",
}
hg38_tracks = config.bigwigs_per_file_assembly[assembly]
track_indices = jnp.asarray([hg38_tracks.index(t) for t in tracks_to_plot.values()])

# token_to_id raises on a name it does not know, rather than returning <unk>.
species_ids = jnp.asarray([species_tokenizer.token_to_id("human")])
print(f"track_indices: {track_indices.tolist()}, species id: {species_ids.tolist()}")

## 4. Predict

In [ ]:
%%time
outs = model(tokens, [species_ids], track_indices=track_indices, skip_lm_head=True)
bigwig = np.asarray(outs["bigwig_tracks_logits"])[0]
print(f"bigwig_tracks_logits: {bigwig.shape}  (positions, selected tracks)")

## 5. Plot

The output covers the central 37.5% of the input window.

In [ ]:
keep = config.keep_target_center_fraction
prediction_start = start + int(window_len * (1 - keep) / 2)
prediction_end = prediction_start + int(window_len * keep)
assert bigwig.shape[0] == prediction_end - prediction_start

fig, axes = plt.subplots(len(tracks_to_plot), 1, figsize=(20, 1.2 * len(tracks_to_plot)), sharex=True)
positions = np.arange(prediction_start, prediction_end)
for column, (ax, title) in enumerate(zip(axes, tracks_to_plot)):
    ax.fill_between(positions, bigwig[:, column])
    ax.set_title(title)
    sns.despine(top=True, right=True, bottom=True, ax=ax)
axes[-1].set_xlabel(f"{chrom}:{prediction_start:,}-{prediction_end:,} ({assembly})")
plt.tight_layout()
plt.show()